In [1]:
# ==========================
# Python Version Check
# ==========================

import sys
from pathlib import Path

MIN_PYTHON_VERSION = (3, 10)


def check_python_version(minimum: tuple = MIN_PYTHON_VERSION) -> None:
    """Ensure the running Python version meets the minimum requirement."""
    current = sys.version_info[:2]

    if current < minimum:
        raise RuntimeError(
            f"DriveWise requires Python {minimum[0]}.{minimum[1]}+, "
            f"but found {current[0]}.{current[1]}."
        )

    print(f"✓ Python Version: {sys.version.split()[0]}")


check_python_version()


# ==========================
# Project Directory Structure
# ==========================

# Automatically detect the project root.
# If the notebook is inside "notebooks/", move one level up.
PROJECT_ROOT = Path.cwd()

if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

# Data directories
DATA_DIR = PROJECT_ROOT / "data"
BROCHURE_DIR = DATA_DIR / "brochures"
PROCESSED_DIR = DATA_DIR / "processed"

# Storage directories
VECTORSTORE_DIR = PROJECT_ROOT / "vectorstore"
LOGS_DIR = PROJECT_ROOT / "logs"
EVALUATION_DIR = PROJECT_ROOT / "evaluation"
APP_DIR = PROJECT_ROOT / "app"

PROJECT_DIRECTORIES = [
    DATA_DIR,
    BROCHURE_DIR,
    PROCESSED_DIR,
    VECTORSTORE_DIR,
    LOGS_DIR,
    EVALUATION_DIR,
    APP_DIR,
]


def create_project_directories(directories: list[Path]) -> None:
    """Create required project directories if they do not already exist."""
    for directory in directories:
        directory.mkdir(parents=True, exist_ok=True)
        print(f"✓ Ready: {directory}")


create_project_directories(PROJECT_DIRECTORIES)

✓ Python Version: 3.12.6
✓ Ready: C:\Users\devan\Desktop\DriveWise\data
✓ Ready: C:\Users\devan\Desktop\DriveWise\data\brochures
✓ Ready: C:\Users\devan\Desktop\DriveWise\data\processed
✓ Ready: C:\Users\devan\Desktop\DriveWise\vectorstore
✓ Ready: C:\Users\devan\Desktop\DriveWise\logs
✓ Ready: C:\Users\devan\Desktop\DriveWise\evaluation
✓ Ready: C:\Users\devan\Desktop\DriveWise\app


In [2]:
# ============================================================
# Standard Library
# ============================================================

import os
import re
import json
import time
import uuid
import hashlib
import warnings

from pathlib import Path
from dataclasses import dataclass

# ============================================================
# Suppress Non-Critical Warnings
# ============================================================

warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=UserWarning)

# ============================================================
# Third-Party Libraries
# ============================================================

import fitz                    # PyMuPDF
import faiss
import numpy as np
import pandas as pd

# Gemini
import google.generativeai as genai

# Sentence Transformers
from sentence_transformers import SentenceTransformer, CrossEncoder

# ============================================================
# Display Settings
# ============================================================

pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", 200)
pd.set_option("display.width", None)

# ============================================================
# Version Information
# ============================================================

print("=" * 60)
print("DriveWise Environment")
print("=" * 60)

print(f"Python : {sys.version.split()[0]}")
print(f"Numpy  : {np.__version__}")
print(f"Pandas : {pd.__version__}")
print(f"FAISS  : {faiss.__version__}")
print(f"PyMuPDF: {fitz.__doc__.split()[1]}")

print("\n✓ All required libraries imported successfully.")

DriveWise Environment
Python : 3.12.6
Numpy  : 2.4.6
Pandas : 3.0.3
FAISS  : 1.14.3
PyMuPDF: 1.28.0:

✓ All required libraries imported successfully.


In [3]:
# ============================================================
# Brochure Discovery
# ============================================================

@dataclass
class BrochureFile:
    """Represents a single brochure discovered in the dataset."""
    
    brand: str
    model: str
    file_path: Path
    document_name: str
    document_version: str = "v1"


def discover_brochures(brochure_dir: Path) -> list[BrochureFile]:
    """
    Discover all brochures stored as:

    data/
        brochures/
            Brand/
                Model.pdf
    """

    if not brochure_dir.exists():
        raise FileNotFoundError(
            f"Brochure directory not found:\n{brochure_dir}"
        )

    brochures = []

    for brand_dir in sorted(brochure_dir.iterdir()):

        if not brand_dir.is_dir():
            continue

        for pdf_path in sorted(brand_dir.glob("*.pdf")):

            brochures.append(
                BrochureFile(
                    brand=brand_dir.name,
                    model=pdf_path.stem,
                    file_path=pdf_path,
                    document_name=pdf_path.name,
                )
            )

    return brochures


# ------------------------------------------------------------
# Discover brochures
# ------------------------------------------------------------

brochures = discover_brochures(BROCHURE_DIR)

if not brochures:
    raise RuntimeError("No brochure PDFs were found inside data/brochures/")

# ------------------------------------------------------------
# Dataset Summary
# ------------------------------------------------------------

brochure_summary = pd.DataFrame(
    [
        {
            "Brand": b.brand,
            "Model": b.model,
            "Document": b.document_name,
            "Version": b.document_version,
            "Path": str(b.file_path),
        }
        for b in brochures
    ]
)

print(f"✓ Successfully discovered {len(brochures)} brochures.\n")

display(brochure_summary)

✓ Successfully discovered 26 brochures.



,Brand,Model,Document,Version,Path
0,HONDA,Amaze,Amaze.pdf,v1,C:\Users\devan\Desktop\DriveWise\data\brochures\HONDA\Amaze.pdf
1,HONDA,City,City.pdf,v1,C:\Users\devan\Desktop\DriveWise\data\brochures\HONDA\City.pdf
2,HONDA,Elevate,Elevate.pdf,v1,C:\Users\devan\Desktop\DriveWise\data\brochures\HONDA\Elevate.pdf
3,HONDA,ZR-V,ZR-V.pdf,v1,C:\Users\devan\Desktop\DriveWise\data\brochures\HONDA\ZR-V.pdf
4,HYUNDAI,alcazar,alcazar.pdf,v1,C:\Users\devan\Desktop\DriveWise\data\brochures\HYUNDAI\alcazar.pdf
5,HYUNDAI,creta,creta.pdf,v1,C:\Users\devan\Desktop\DriveWise\data\brochures\HYUNDAI\creta.pdf
6,HYUNDAI,i20,i20.pdf,v1,C:\Users\devan\Desktop\DriveWise\data\brochures\HYUNDAI\i20.pdf
7,HYUNDAI,ioniq-5,ioniq-5.pdf,v1,C:\Users\devan\Desktop\DriveWise\data\brochures\HYUNDAI\ioniq-5.pdf
8,HYUNDAI,venue,venue.pdf,v1,C:\Users\devan\Desktop\DriveWise\data\brochures\HYUNDAI\venue.pdf
9,HYUNDAI,verna,verna.pdf,v1,C:\Users\devan\Desktop\DriveWise\data\brochures\HYUNDAI\verna.pdf


In [4]:
# ============================================================
# Page-Level Metadata Container
# ============================================================
@dataclass
class ExtractedPage:
    """Represents a single extracted brochure page, tagged with source metadata."""

    brand: str
    model: str
    document_name: str
    document_version: str
    page_number: int
    text: str


# ============================================================
# PDF Text Extraction
# ============================================================
def extract_brochure_pages(brochure: BrochureFile) -> list[ExtractedPage]:
    """
    Open a brochure PDF and extract raw text from every page, along with
    source metadata (brand, model, document, page number). No chunking or
    section logic happens here — that's handled in Section 04.
    """
    pages = []
    try:
        with fitz.open(brochure.file_path) as doc:
            for i, page in enumerate(doc):
                pages.append(
                    ExtractedPage(
                        brand=brochure.brand,
                        model=brochure.model,
                        document_name=brochure.document_name,
                        document_version=brochure.document_version,
                        page_number=i + 1,
                        text=page.get_text().strip(),
                    )
                )
    except Exception as exc:
        print(f"✗ Failed to read {brochure.document_name}: {exc}")
        return []

    return pages


# ------------------------------------------------------------
# Extract all pages from all discovered brochures
# ------------------------------------------------------------
all_pages: list[ExtractedPage] = []
for brochure in brochures:
    all_pages.extend(extract_brochure_pages(brochure))

if not all_pages:
    raise RuntimeError("No pages were extracted from the discovered brochures.")

# ============================================================
# Extraction Summary
# ============================================================
total_brochures = len({(p.brand, p.model) for p in all_pages})
total_pages = len(all_pages)
total_chars = sum(len(p.text) for p in all_pages)
empty_pages = sum(1 for p in all_pages if not p.text)

print(f"✓ Brochures processed: {total_brochures}")
print(f"✓ Pages extracted: {total_pages}")
print(f"✓ Total characters extracted: {total_chars:,}")
print(f"✓ Empty pages: {empty_pages}")

# ------------------------------------------------------------
# Preview
# ------------------------------------------------------------
extraction_summary = pd.DataFrame(
    [
        {
            "Brand": p.brand,
            "Model": p.model,
            "Page": p.page_number,
            "Chars": len(p.text),
        }
        for p in all_pages
    ]
)
display(extraction_summary.head())

✓ Brochures processed: 26
✓ Pages extracted: 468
✓ Total characters extracted: 430,066
✓ Empty pages: 19


,Brand,Model,Page,Chars
0,HONDA,Amaze,1,0
1,HONDA,Amaze,2,1493
2,HONDA,Amaze,3,4586
3,HONDA,City,1,0
4,HONDA,City,2,230


In [5]:
# ============================================================
# Section Classification
# ============================================================
SECTION_KEYWORDS = {
    "engine_performance": ["engine", "performance", "power", "torque", "motor", "acceleration"],
    "mileage_efficiency": ["mileage", "range", "km/charge", "fuel efficiency", "kmpl"],
    "safety": ["safety", "airbag", "abs", "adas", "collision", "brake"],
    "dimensions": ["dimensions", "wheelbase", "ground clearance", "length", "width", "height", "boot space"],
    "interior_comfort": ["interior", "comfort", "seat", "upholstery", "ambient lighting"],
    "exterior_styling": ["exterior", "styling", "wheel", "alloy", "led", "sunroof"],
    "infotainment_connectivity": ["infotainment", "touchscreen", "connect", "app", "bluetooth", "speaker", "charging"],
    "features": ["feature", "variant", "trim"],
    "legal_disclaimer": ["terms & conditions", "reserves the right", "warranty", "disclaims"],
}


def classify_section(text: str) -> str:
    """
    Keyword-frequency section classifier. Chosen over an LLM-based classifier
    to keep chunking deterministic, fast, and free to run on every page at
    ingestion time.
    """
    text_lower = text.lower()
    scores = {sec: sum(text_lower.count(kw) for kw in kws) for sec, kws in SECTION_KEYWORDS.items()}
    best_section = max(scores, key=scores.get)
    return best_section if scores[best_section] > 0 else "general"


# ============================================================
# Chunk Container
# ============================================================
@dataclass
class Chunk:
    """A single retrievable unit of brochure content, tagged with metadata."""

    chunk_id: str
    brand: str
    model: str
    document_name: str
    document_version: str
    page: int
    section: str
    text: str


def make_chunk_id(brand: str, model: str, page: int) -> str:
    raw = f"{brand}|{model}|{page}"
    return hashlib.md5(raw.encode()).hexdigest()[:12]


# ============================================================
# Chunk Builder
# ============================================================
def build_chunks_from_pages(pages: list[ExtractedPage]) -> list[Chunk]:
    """
    One chunk per page, built directly from Section 03's extracted pages
    (no re-opening PDFs). Only completely empty pages are skipped — short
    pages (e.g. a cover page) are kept as-is for now. We'll split chunks
    further only if evaluation later shows page-level granularity hurts
    retrieval precision.
    """
    chunks = []
    for page in pages:
        if not page.text:
            continue
        section = classify_section(page.text)
        chunks.append(
            Chunk(
                chunk_id=make_chunk_id(page.brand, page.model, page.page_number),
                brand=page.brand,
                model=page.model,
                document_name=page.document_name,
                document_version=page.document_version,
                page=page.page_number,
                section=section,
                text=page.text,
            )
        )
    return chunks


# ------------------------------------------------------------
# Build chunks from all extracted pages
# ------------------------------------------------------------
chunks: list[Chunk] = build_chunks_from_pages(all_pages)

if not chunks:
    raise RuntimeError("No chunks were built from the extracted pages.")

# ============================================================
# Chunking Summary
# ============================================================
chunk_lengths = [len(c.text) for c in chunks]

print(f"✓ Total chunks: {len(chunks)}")
print(f"✓ Average chunk length: {sum(chunk_lengths) / len(chunk_lengths):.0f} chars")
print("✓ Section distribution:")
print(pd.Series([c.section for c in chunks]).value_counts())

# ------------------------------------------------------------
# Preview
# ------------------------------------------------------------
chunk_df = pd.DataFrame(
    [
        {
            "Brand": c.brand,
            "Model": c.model,
            "Page": c.page,
            "Section": c.section,
            "Chars": len(c.text),
            "Text": c.text,
        }
        for c in chunks
    ]
)
display(chunk_df.head())

✓ Total chunks: 449
✓ Average chunk length: 958 chars
✓ Section distribution:
interior_comfort             80
exterior_styling             72
infotainment_connectivity    72
safety                       68
engine_performance           65
general                      64
dimensions                   11
features                      7
mileage_efficiency            6
legal_disclaimer              4
Name: count, dtype: int64


,Brand,Model,Page,Section,Chars,Text
0,HONDA,Amaze,2,exterior_styling,1493,Disclaimer:*As certified under Rule 115 of CMVR 1989 \n \n \n \nThe spare wheel does not come with Alloy Wheel and the rim / tyre size may differ from the standard size depending on the variant \n...
1,HONDA,Amaze,3,interior_comfort,4586,"Amaze S* (Standard Features)\nDisclaimer:\nColors, features and specifications are grade specific\nAmaze S Grade refers to AMAZE RDE S MT/CVT R (i-VTEC)\nTerms & Conditions apply\nTHE DISTINCTION..."
2,HONDA,City,2,infotainment_connectivity,230,One look and you’ll feel\na connection that matches your energy.\nIt’s less about the drive and more about the feeling.\nIt's everything you ever dreamt of.\nMeet the new Honda City.\nAll Feels. A...
3,HONDA,City,3,exterior_styling,44,Exterior\nTurn Heads with\na Striking New Vibe
4,HONDA,City,4,exterior_styling,223,Aero-Blade Diamond Cut Alloy Wheels\nSharp Front Design with Blade-Eye \nSignature LED Lighting System\nSignature Matrix-Mesh Lower\nBumper Garnish \nSporty Trunk Lip Spoiler\nSignature Z-Edge Wr...


In [6]:
# ============================================================
# Embedding Model
# ============================================================
EMBEDDING_MODEL_NAME = "BAAI/bge-small-en-v1.5"   # 384-dim, stronger retrieval quality than MiniLM

try:
    embedder = SentenceTransformer(EMBEDDING_MODEL_NAME)
except Exception as exc:
    raise RuntimeError(
        f"Failed to load embedding model '{EMBEDDING_MODEL_NAME}'. "
        f"Check your internet connection or Hugging Face access.\nOriginal error: {exc}"
    )

EMBEDDING_DIM = embedder.get_sentence_embedding_dimension()


# ============================================================
# Embedding Generation
# ============================================================
def embed_chunks(chunks: list[Chunk]) -> np.ndarray:
    """
    Encode all chunk texts into normalized embedding vectors. Normalizing here
    (rather than at search time) means the vector store can use a plain inner
    product for cosine similarity later.

    Note: BGE models recommend prefixing QUERIES (not corpus documents) with
    an instruction string for best retrieval quality — e.g.
    "Represent this sentence for searching relevant passages: {query}".
    That belongs in Section 08 (Retriever), not here — chunk/document text is
    embedded as-is.
    """
    texts = [c.text for c in chunks]
    embeddings = embedder.encode(texts, convert_to_numpy=True, show_progress_bar=True)
    faiss.normalize_L2(embeddings)
    return embeddings.astype("float32")


# ------------------------------------------------------------
# Generate embeddings for all chunks (timed)
# ------------------------------------------------------------
start_time = time.perf_counter()
chunk_embeddings = embed_chunks(chunks)
elapsed_time = time.perf_counter() - start_time

if chunk_embeddings.shape[0] != len(chunks):
    raise RuntimeError("Embedding count does not match chunk count.")

# ============================================================
# Embedding Summary
# ============================================================
print(f"✓ Embedding model: {EMBEDDING_MODEL_NAME}")
print(f"✓ Chunks embedded: {chunk_embeddings.shape[0]}")
print(f"✓ Embedding dimension: {EMBEDDING_DIM}")
print(f"✓ Time taken: {elapsed_time:.2f}s")
print(f"✓ Vector norm check (should be ~1.0): {np.linalg.norm(chunk_embeddings[0]):.4f}")

Batches:   0%|          | 0/15 [00:00<?, ?it/s]

✓ Embedding model: BAAI/bge-small-en-v1.5
✓ Chunks embedded: 449
✓ Embedding dimension: 384
✓ Time taken: 19.17s
✓ Vector norm check (should be ~1.0): 1.0000


In [7]:
# ============================================================
# FAISS Index Construction
# ============================================================
def build_faiss_index(embeddings: np.ndarray, dim: int) -> faiss.Index:
    """
    Flat inner-product index. Embeddings are already L2-normalized (Section 05),
    so inner product is equivalent to cosine similarity. IndexFlatIP is exact
    (no approximation) — appropriate given per-project chunk counts are small
    (tens to low thousands, not millions).
    """
    if embeddings.shape[1] != dim:
        raise ValueError(f"Embedding dimension mismatch: expected {dim}, got {embeddings.shape[1]}")

    index = faiss.IndexFlatIP(dim)
    index.add(embeddings)
    return index


try:
    faiss_index = build_faiss_index(chunk_embeddings, EMBEDDING_DIM)
except Exception as exc:
    raise RuntimeError(f"Failed to build FAISS index: {exc}")


# ============================================================
# Save / Load Vector Store
# ============================================================
def save_vector_store(index: faiss.Index, chunks: list[Chunk], directory: Path) -> None:
    """Persist the FAISS index and its aligned chunk metadata to disk."""
    try:
        faiss.write_index(index, str(directory / "index.faiss"))
        metadata_df = pd.DataFrame(
            [
                {
                    "chunk_id": c.chunk_id,
                    "brand": c.brand,
                    "model": c.model,
                    "document_name": c.document_name,
                    "document_version": c.document_version,
                    "page": c.page,
                    "section": c.section,
                    "text": c.text,
                }
                for c in chunks
            ]
        )
        metadata_df.to_csv(directory / "chunk_metadata.csv", index=False)
    except Exception as exc:
        raise RuntimeError(f"Failed to save vector store to {directory}: {exc}")


def load_vector_store(directory: Path) -> tuple[faiss.Index, pd.DataFrame]:
    """Load a previously saved FAISS index and its chunk metadata."""
    try:
        index = faiss.read_index(str(directory / "index.faiss"))
        metadata_df = pd.read_csv(directory / "chunk_metadata.csv")
    except Exception as exc:
        raise RuntimeError(
            f"Failed to load vector store from {directory}: {exc}"
        )

    return index, metadata_df


# ------------------------------------------------------------
# Save the vector store
# ------------------------------------------------------------
save_vector_store(faiss_index, chunks, VECTORSTORE_DIR)

# Keep a metadata DataFrame in memory too, aligned by row position with
# faiss_index (row i in the index <-> row i in chunk_metadata_df). Later
# sections filter/retrieve against this DataFrame.
chunk_metadata_df = pd.DataFrame(
    [
        {
            "chunk_id": c.chunk_id,
            "brand": c.brand,
            "model": c.model,
            "document_name": c.document_name,
            "document_version": c.document_version,
            "page": c.page,
            "section": c.section,
            "text": c.text,
        }
        for c in chunks
    ]
)

# ============================================================
# Vector Store Summary
# ============================================================
print(f"✓ FAISS index type: {type(faiss_index).__name__}")
print(f"✓ Vectors indexed: {faiss_index.ntotal}")
print(f"✓ Index dimension: {EMBEDDING_DIM}")
print(f"✓ Saved to: {VECTORSTORE_DIR}")

if faiss_index.ntotal != len(chunk_metadata_df):
    raise RuntimeError("Mismatch between FAISS index size and chunk metadata rows.")
print("✓ Index size matches chunk metadata rows.")

✓ FAISS index type: IndexFlatIP
✓ Vectors indexed: 449
✓ Index dimension: 384
✓ Saved to: C:\Users\devan\Desktop\DriveWise\vectorstore
✓ Index size matches chunk metadata rows.


In [8]:
# ============================================================
# Metadata Filtering
# ============================================================
def filter_chunk_indices(metadata: pd.DataFrame, brand: str, model: str) -> np.ndarray:
    """
    Return row indices (aligned with faiss_index / chunk_metadata_df rows)
    matching the selected brand + model. This runs BEFORE any vector search,
    so similarity search only ever operates over the selected vehicle's chunks.
    """
    if "brand" not in metadata.columns or "model" not in metadata.columns:
        raise KeyError("Metadata DataFrame is missing required 'brand'/'model' columns.")

    mask = (metadata["brand"] == brand) & (metadata["model"] == model)
    matched_indices = metadata[mask].index.to_numpy()

    if matched_indices.size == 0:
        available = metadata[["brand", "model"]].drop_duplicates().values.tolist()
        print(f"⚠ No chunks found for brand='{brand}', model='{model}'.")
        print(f"  Available brand/model pairs: {available}")

    return matched_indices


# ============================================================
# Available Brand / Model Options
# ============================================================
def get_available_brands(metadata: pd.DataFrame) -> list[str]:
    """Return sorted unique brands — used to populate a brand selector."""
    return sorted(metadata["brand"].unique().tolist())


def get_available_models(metadata: pd.DataFrame, brand: str) -> list[str]:
    """Return sorted unique models for a given brand — used to populate a model selector."""
    return sorted(metadata[metadata["brand"] == brand]["model"].unique().tolist())


# ------------------------------------------------------------
# Sanity check against the actual chunk metadata
# ------------------------------------------------------------
available_brands = get_available_brands(chunk_metadata_df)
print(f"✓ Available brands: {available_brands}")

for brand in available_brands:
    models = get_available_models(chunk_metadata_df, brand)
    print(f"  - {brand}: {models}")

sample_brand = available_brands[0]
sample_model = get_available_models(chunk_metadata_df, sample_brand)[0]
sample_indices = filter_chunk_indices(chunk_metadata_df, sample_brand, sample_model)

print(f"\n✓ Filter test: brand='{sample_brand}', model='{sample_model}' -> {len(sample_indices)} chunk(s)")

# Deliberately test the "no match" warning path too
_ = filter_chunk_indices(chunk_metadata_df, "NonExistentBrand", "NonExistentModel")

✓ Available brands: ['HONDA', 'HYUNDAI', 'KIA', 'TATA', 'TOYOTA']
  - HONDA: ['Amaze', 'City', 'Elevate', 'ZR-V']
  - HYUNDAI: ['alcazar', 'creta', 'i20', 'ioniq-5', 'venue', 'verna']
  - KIA: ['Carnival', 'Clavis', 'EV6', 'EV9', 'seltos', 'syros']
  - TATA: ['Nexon', 'harrier', 'punch', 'safari', 'sierra']
  - TOYOTA: ['glanza', 'hilux', 'hycross', 'innova-crysta', 'vellfire']

✓ Filter test: brand='HONDA', model='Amaze' -> 2 chunk(s)
⚠ No chunks found for brand='NonExistentBrand', model='NonExistentModel'.
  Available brand/model pairs: [['HONDA', 'Amaze'], ['HONDA', 'City'], ['HONDA', 'Elevate'], ['HONDA', 'ZR-V'], ['HYUNDAI', 'alcazar'], ['HYUNDAI', 'creta'], ['HYUNDAI', 'i20'], ['HYUNDAI', 'ioniq-5'], ['HYUNDAI', 'venue'], ['HYUNDAI', 'verna'], ['KIA', 'Carnival'], ['KIA', 'Clavis'], ['KIA', 'EV6'], ['KIA', 'EV9'], ['KIA', 'seltos'], ['KIA', 'syros'], ['TATA', 'harrier'], ['TATA', 'Nexon'], ['TATA', 'punch'], ['TATA', 'safari'], ['TATA', 'sierra'], ['TOYOTA', 'glanza'], ['TOYOTA',

In [9]:
# ============================================================
# Query Embedding
# ============================================================
# BGE models are trained with an instruction prefix on the QUERY side only
# (corpus/document text is embedded as-is, per Section 05). This measurably
# improves retrieval quality for bge-* models.
BGE_QUERY_INSTRUCTION = "Represent this sentence for searching relevant passages: "


def embed_query(query: str) -> np.ndarray:
    """Encode a single user query into a normalized embedding vector."""
    prefixed_query = BGE_QUERY_INSTRUCTION + query
    query_vec = embedder.encode([prefixed_query], convert_to_numpy=True)
    faiss.normalize_L2(query_vec)
    return query_vec.astype("float32")   # shape (1, dim) — what faiss_index.search expects


# ============================================================
# FAISS-Based Retrieval with Metadata Filtering
# ============================================================
def retrieve(
    query: str,
    brand: str,
    model: str,
    index: faiss.Index,
    metadata: pd.DataFrame,
    top_k: int = 10,
) -> tuple[pd.DataFrame, float]:
    """
    Search the FAISS index directly, then filter results down to the
    selected brand/model. Because IndexFlatIP holds all chunks across all
    brochures, we over-fetch candidates (search the full index, since our
    current chunk counts are small) so filtering doesn't leave us short of
    top_k matches — then truncate to top_k after filtering, preserving
    FAISS's similarity ranking.
    """
    allowed_ids = set(filter_chunk_indices(metadata, brand, model).tolist())
    if not allowed_ids:
        return pd.DataFrame(columns=list(metadata.columns) + ["similarity_score"]), 0.0

    query_vec = embed_query(query)

    start_time = time.perf_counter()
    search_k = index.ntotal   # search everything; dataset is small enough that this is cheap
    scores, indices = index.search(query_vec, search_k)
    retrieval_time = time.perf_counter() - start_time

    scores, indices = scores[0], indices[0]

    filtered = [(idx, score) for idx, score in zip(indices, scores) if idx in allowed_ids]
    filtered = filtered[:top_k]

    if not filtered:
        return pd.DataFrame(columns=list(metadata.columns) + ["similarity_score"]), retrieval_time

    matched_rows, matched_scores = zip(*filtered)
    result = metadata.loc[list(matched_rows)].copy()
    result["similarity_score"] = matched_scores
    return result.reset_index(drop=True), retrieval_time


# ------------------------------------------------------------
# Sanity check against real data
# ------------------------------------------------------------
sample_query = "What is the safety rating and airbag count?"
retrieved, retrieval_time = retrieve(sample_query, sample_brand, sample_model, faiss_index, chunk_metadata_df, top_k=5)

print(f"✓ Query: '{sample_query}'")
print(f"✓ Brand/Model: {sample_brand} / {sample_model}")
print(f"✓ Chunks retrieved: {len(retrieved)}")
print(f"✓ Retrieval time: {retrieval_time*1000:.2f} ms")
if not retrieved.empty:
    print(f"✓ Top similarity score: {retrieved['similarity_score'].iloc[0]:.4f}")
display(retrieved[["page", "section", "similarity_score"]].head())

✓ Query: 'What is the safety rating and airbag count?'
✓ Brand/Model: HONDA / Amaze
✓ Chunks retrieved: 2
✓ Retrieval time: 0.17 ms
✓ Top similarity score: 0.5730


,page,section,similarity_score
0,3,interior_comfort,0.573008
1,2,exterior_styling,0.544548


In [10]:
# ============================================================
# Cross-Encoder Re-ranker
# ============================================================
RERANKER_MODEL_NAME = "cross-encoder/ms-marco-MiniLM-L-6-v2"

try:
    reranker = CrossEncoder(RERANKER_MODEL_NAME)
except Exception as exc:
    raise RuntimeError(
        f"Failed to load re-ranker model '{RERANKER_MODEL_NAME}'. "
        f"Check your internet connection or Hugging Face access.\nOriginal error: {exc}"
    )


# ============================================================
# Re-ranking + Context Window Control
# ============================================================
def rerank(query: str, retrieved: pd.DataFrame, top_n: int = 4) -> tuple[pd.DataFrame, float]:
    """
    Score each retrieved chunk against the query with a cross-encoder (which
    jointly attends to query+passage, unlike the bi-encoder similarity used
    for initial retrieval) and keep only the top_n — this is the context
    window control step, reducing prompt noise before generation.
    """
    if retrieved.empty:
        return retrieved, 0.0

    pairs = [(query, row.text) for row in retrieved.itertuples()]

    start_time = time.perf_counter()
    scores = reranker.predict(pairs)
    rerank_time = time.perf_counter() - start_time

    result = retrieved.copy()
    result["rerank_score"] = scores
    result = result.sort_values("rerank_score", ascending=False).head(top_n)
    return result.reset_index(drop=True), rerank_time


# ------------------------------------------------------------
# Sanity check against real data
# ------------------------------------------------------------
reranked, rerank_time = rerank(sample_query, retrieved, top_n=4)

print(f"✓ Query: '{sample_query}'")
print(f"✓ Chunks in: {len(retrieved)} -> Chunks out: {len(reranked)}")
print(f"✓ Re-ranking time: {rerank_time*1000:.2f} ms")
if not reranked.empty:
    print(f"✓ Top rerank score: {reranked['rerank_score'].iloc[0]:.4f}")
display(reranked[["page", "section", "similarity_score", "rerank_score"]].head())

✓ Query: 'What is the safety rating and airbag count?'
✓ Chunks in: 2 -> Chunks out: 2
✓ Re-ranking time: 121.65 ms
✓ Top rerank score: -7.5077


,page,section,similarity_score,rerank_score
0,3,interior_comfort,0.573008,-7.507675
1,2,exterior_styling,0.544548,-11.158991


In [11]:
# ============================================================
# Prompt Template
# ============================================================
PROMPT_TEMPLATE = """You are DriveWise, an assistant that answers questions about the {brand} {model} \
using ONLY the brochure excerpts provided below.

Rules:
- Answer strictly from the provided context. Do not use outside knowledge.
- If the context does not contain the answer, say so explicitly — do not guess or hallucinate.
- Keep the answer concise and directly relevant to the question.
- When useful, mention which section/page the information came from.

Context:
{context}

Question: {question}

Answer:"""


# ============================================================
# Context Formatting
# ============================================================
def format_context(context_chunks: pd.DataFrame) -> str:
    """Format re-ranked chunks into labeled context blocks for the prompt."""
    blocks = []
    for row in context_chunks.itertuples():
        blocks.append(f"[Section: {row.section} | Page {row.page}]\n{row.text}")
    return "\n\n".join(blocks)


def build_prompt(question: str, brand: str, model: str, context_chunks: pd.DataFrame) -> str:
    """
    Assemble the final prompt sent to the LLM. Raises if there's no context —
    generation should never be attempted on an empty context window, since
    that's exactly the condition the "answer only from context" rule is meant
    to prevent hallucination against.
    """
    if context_chunks.empty:
        raise ValueError("Cannot build a prompt with no retrieved context.")

    context = format_context(context_chunks)
    return PROMPT_TEMPLATE.format(brand=brand, model=model, context=context, question=question)


# ------------------------------------------------------------
# Sanity check against real data
# ------------------------------------------------------------
sample_prompt = build_prompt(sample_query, sample_brand, sample_model, reranked)

print(f"✓ Prompt built for: {sample_brand} {sample_model}")
print(f"✓ Context chunks included: {len(reranked)}")
print(f"✓ Prompt length: {len(sample_prompt)} chars")
print("\n--- Prompt Preview ---\n")
print(sample_prompt[:800] + ("..." if len(sample_prompt) > 800 else ""))

✓ Prompt built for: HONDA Amaze
✓ Context chunks included: 2
✓ Prompt length: 6667 chars

--- Prompt Preview ---

You are DriveWise, an assistant that answers questions about the HONDA Amaze using ONLY the brochure excerpts provided below.

Rules:
- Answer strictly from the provided context. Do not use outside knowledge.
- If the context does not contain the answer, say so explicitly — do not guess or hallucinate.
- Keep the answer concise and directly relevant to the question.
- When useful, mention which section/page the information came from.

Context:
[Section: interior_comfort | Page 3]
Amaze S* (Standard Features)
Disclaimer:
Colors, features and specifications are grade specific
Amaze S Grade refers to AMAZE RDE S MT/CVT  R (i-VTEC)
Terms & Conditions apply
THE DISTINCTION
OF GREATNESS
Terms and conditions applicable. The Images and depictions shown in this brochure are computer generated/ enha...


In [12]:
# ============================================================
# LLM Configuration
# ============================================================

from dotenv import load_dotenv
import os

# Current Gemini model used throughout the project.
# Change only this line if you want to switch models later.
LLM_MODEL_NAME = "gemini-3.1-flash-lite"# Load environment variables
load_dotenv()

GOOGLE_API_KEY = os.getenv("GOOGLE_API_KEY")

if not GOOGLE_API_KEY:
    raise RuntimeError(
        "GOOGLE_API_KEY not found.\n"
        "Create a .env file in the project root containing:\n"
        "GOOGLE_API_KEY=your_api_key"
    )

try:
    genai.configure(api_key=GOOGLE_API_KEY)
    llm = genai.GenerativeModel(LLM_MODEL_NAME)

except Exception as exc:
    raise RuntimeError(f"Failed to configure Gemini: {exc}")

print(f"✓ Gemini model loaded: {LLM_MODEL_NAME}")


# ============================================================
# RAG Response Container
# ============================================================

@dataclass
class RagResponse:
    answer: str
    sources: list[dict]
    retrieval_time: float
    rerank_time: float
    generation_time: float
    status: str


# ============================================================
# End-to-End RAG Pipeline
# ============================================================

def ask_drivewise(
    question: str,
    brand: str,
    model: str,
    index: faiss.Index,
    metadata: pd.DataFrame,
    top_k: int = 10,
    top_n: int = 4,
) -> RagResponse:

    retrieved, retrieval_time = retrieve(
        question,
        brand,
        model,
        index,
        metadata,
        top_k=top_k,
    )

    if retrieved.empty:
        return RagResponse(
            answer=f"No brochure data found for {brand} {model}.",
            sources=[],
            retrieval_time=retrieval_time,
            rerank_time=0.0,
            generation_time=0.0,
            status="no_context",
        )

    reranked, rerank_time = rerank(
        question,
        retrieved,
        top_n=top_n,
    )

    prompt = build_prompt(
        question,
        brand,
        model,
        reranked,
    )

    start_time = time.perf_counter()

    try:
        response = llm.generate_content(prompt)

        # Safely extract the generated text
        answer = response.text.strip() if hasattr(response, "text") else "No response generated."
        status = "success"

    except Exception as exc:
        answer = f"Generation failed: {exc}"
        status = "error"

    generation_time = time.perf_counter() - start_time

    sources = [
        {
            "document": row.document_name,
            "page": int(row.page),
            "section": row.section,
        }
        for row in reranked.itertuples()
    ]

    return RagResponse(
        answer=answer,
        sources=sources,
        retrieval_time=retrieval_time,
        rerank_time=rerank_time,
        generation_time=generation_time,
        status=status,
    )


# ------------------------------------------------------------
# Sanity Check
# ------------------------------------------------------------

response = ask_drivewise(
    sample_query,
    sample_brand,
    sample_model,
    faiss_index,
    chunk_metadata_df,
)

print(f"✓ Model: {LLM_MODEL_NAME}")
print(f"✓ Status: {response.status}")
print(f"✓ Retrieval: {response.retrieval_time*1000:.2f} ms")
print(f"✓ Rerank: {response.rerank_time*1000:.2f} ms")
print(f"✓ Generation: {response.generation_time*1000:.2f} ms")
print(f"✓ Sources: {len(response.sources)}")

print("\nAnswer:\n")
print(response.answer)

✓ Gemini model loaded: gemini-3.1-flash-lite
✓ Model: gemini-3.1-flash-lite
✓ Status: success
✓ Retrieval: 0.12 ms
✓ Rerank: 106.47 ms
✓ Generation: 1567.75 ms
✓ Sources: 2

Answer:

The provided brochure does not contain information regarding the safety rating or the specific airbag count for the Honda Amaze. It only lists the "Driver Seat i-SRS Airbag System" and "Passenger Seat SRS Airbag System" under the safety features.


In [13]:
# ============================================================
# Log File
# ============================================================
LOG_FILE = LOGS_DIR / "query_log.jsonl"


# ============================================================
# Interaction Logging
# ============================================================
def log_interaction(
    question: str,
    brand: str,
    model: str,
    response: RagResponse,
) -> dict:
    """Append one query/response record to a JSONL log."""

    entry = {
        "timestamp": time.strftime("%Y-%m-%d %H:%M:%S"),
        "query": question,
        "brand": brand,
        "model": model,
        "response_time_s": round(
            response.retrieval_time +
            response.rerank_time +
            response.generation_time,
            4,
        ),
        "retrieval_latency_s": round(response.retrieval_time, 4),
        "rerank_latency_s": round(response.rerank_time, 4),
        "generation_latency_s": round(response.generation_time, 4),
        "retrieved_chunks": len(response.sources),
        "sources": response.sources,
        "status": response.status,
    }

    try:
        with open(LOG_FILE, "a", encoding="utf-8") as f:
            json.dump(entry, f, ensure_ascii=False)
            f.write("\n")
    except Exception as exc:
        print(f"⚠ Failed to write log: {exc}")

    return entry


# ============================================================
# Load Logs
# ============================================================
def load_query_log(log_file: Path = LOG_FILE) -> pd.DataFrame:
    """Load the JSONL interaction log."""

    if not log_file.exists():
        return pd.DataFrame()

    try:
        with open(log_file, "r", encoding="utf-8") as f:
            records = [json.loads(line) for line in f]
        return pd.DataFrame(records)
    except Exception as exc:
        print(f"⚠ Failed to read log: {exc}")
        return pd.DataFrame()


# ------------------------------------------------------------
# Log Sample Query
# ------------------------------------------------------------
log_entry = log_interaction(
    sample_query,
    sample_brand,
    sample_model,
    response,
)

print(f"✓ Logged to: {LOG_FILE}")
print(f"✓ Status: {log_entry['status']}")
print(f"✓ Response Time: {log_entry['response_time_s']} s")

log_df = load_query_log()

print(f"\n✓ Total Logged Queries: {len(log_df)}")

if not log_df.empty:
    display(log_df.tail())

✓ Logged to: C:\Users\devan\Desktop\DriveWise\logs\query_log.jsonl
✓ Status: success
✓ Response Time: 1.6743 s

✓ Total Logged Queries: 8


,timestamp,query,brand,model,response_time_s,retrieval_latency_s,rerank_latency_s,generation_latency_s,retrieved_chunks,sources,status
3,2026-07-12 18:40:58,What is the safety rating and airbag count?,HONDA,Amaze,1.0139,0.0004,0.4507,0.5628,2,"[{'document': 'Amaze.pdf', 'page': 3, 'section': 'interior_comfort'}, {'document': 'Amaze.pdf', 'page': 2, 'section': 'exterior_styling'}]",error
4,2026-07-12 18:52:49,What is the safety rating and airbag count?,HONDA,Amaze,1.2669,0.0003,0.4413,0.8252,2,"[{'document': 'Amaze.pdf', 'page': 3, 'section': 'interior_comfort'}, {'document': 'Amaze.pdf', 'page': 2, 'section': 'exterior_styling'}]",error
5,2026-07-13 00:24:40,What is the safety rating and airbag count?,HONDA,Amaze,26.5253,0.0001,0.1234,26.4017,2,"[{'document': 'Amaze.pdf', 'page': 3, 'section': 'interior_comfort'}, {'document': 'Amaze.pdf', 'page': 2, 'section': 'exterior_styling'}]",success
6,2026-07-13 02:30:40,What is the safety rating and airbag count?,HONDA,Amaze,1.6905,0.0001,0.1170,1.5733,2,"[{'document': 'Amaze.pdf', 'page': 3, 'section': 'interior_comfort'}, {'document': 'Amaze.pdf', 'page': 2, 'section': 'exterior_styling'}]",success
7,2026-07-13 02:34:09,What is the safety rating and airbag count?,HONDA,Amaze,1.6743,0.0001,0.1065,1.5678,2,"[{'document': 'Amaze.pdf', 'page': 3, 'section': 'interior_comfort'}, {'document': 'Amaze.pdf', 'page': 2, 'section': 'exterior_styling'}]",success


In [14]:
# ============================================================
# Lightweight Evaluation (no external frameworks)
# ============================================================
# 5-10 brochure-agnostic questions — phrased generally enough to apply to
# any selected brand/model, not just EVs.
EVAL_QUESTIONS = [
    "What is the mileage or fuel efficiency of this car?",
    "How many standard safety features does the car have?",
    "What is the engine or motor's power output?",
    "What is the seating capacity?",
    "What are the exterior dimensions of the car?",
    "What infotainment features are available?",
    "What is the warranty offered on this vehicle?",
    "What variants or trims are available?",
]

eval_records = []

for question in EVAL_QUESTIONS:
    response = ask_drivewise(question, sample_brand, sample_model, faiss_index, chunk_metadata_df)

    total_latency = response.retrieval_time + response.rerank_time + response.generation_time

    eval_records.append({
        "Question": question,
        "Status": response.status,
        "Retrieval (s)": round(response.retrieval_time, 4),
        "Rerank (s)": round(response.rerank_time, 4),
        "Generation (s)": round(response.generation_time, 4),
        "Total (s)": round(total_latency, 4),
        "Sources Retrieved": len(response.sources),
    })

eval_df = pd.DataFrame(eval_records)
display(eval_df)

# ============================================================
# Summary
# ============================================================
avg_retrieval = eval_df["Retrieval (s)"].mean()
avg_rerank = eval_df["Rerank (s)"].mean()
avg_generation = eval_df["Generation (s)"].mean()
avg_total = eval_df["Total (s)"].mean()
success_rate = (eval_df["Status"] == "success").mean() * 100

print("✓ Evaluation Summary")
print(f"  Questions evaluated: {len(eval_df)}")
print(f"  Success rate: {success_rate:.1f}%")
print(f"  Avg retrieval latency: {avg_retrieval:.4f}s")
print(f"  Avg rerank latency: {avg_rerank:.4f}s")
print(f"  Avg generation latency: {avg_generation:.4f}s")
print(f"  Avg total latency: {avg_total:.4f}s")

,Question,Status,Retrieval (s),Rerank (s),Generation (s),Total (s),Sources Retrieved
0,What is the mileage or fuel efficiency of this car?,success,0.0001,0.1029,0.8629,0.9659,2
1,How many standard safety features does the car have?,success,0.0001,0.1096,1.1340,1.2437,2
2,What is the engine or motor's power output?,success,0.0001,0.1096,0.7210,0.8308,2
3,What is the seating capacity?,success,0.0001,0.1119,0.8344,0.9463,2
4,What are the exterior dimensions of the car?,success,0.0001,0.1087,0.8185,0.9273,2
5,What infotainment features are available?,success,0.0001,0.1092,1.0088,1.1181,2
6,What is the warranty offered on this vehicle?,success,0.0001,0.1133,0.8444,0.9578,2
7,What variants or trims are available?,success,0.0001,0.1115,1.0105,1.1222,2


✓ Evaluation Summary
  Questions evaluated: 8
  Success rate: 100.0%
  Avg retrieval latency: 0.0001s
  Avg rerank latency: 0.1096s
  Avg generation latency: 0.9043s
  Avg total latency: 1.0140s


## 14 | Conclusion

### Project Overview
DriveWise is a metadata-aware and brochure-based conversational assistant answering
questions in natural language regarding particular brands and models of cars based on
Retrieval-Augmented Generation. The answers are generated by means of a language model
based on the retrieved information from brochures only and are always attributed back
to the source document, section, and page number.

### What Was Implemented
The notebook implements the whole pipeline gradually: brochure search and extraction of
the metadata from PDF files, chunking by means of keyphrases and classification into
sections, dense vectors creation via BAAI/bge-small-en-v1.5, FAISS IndexFlatIP
vector storage, metadata-based retrieval limited to the specified brand and model,
cross-encoder re-ranking for controlling context window, the template for creating a
prompt focused on grounding, orchestration via Gemini, structured logging in JSONL,
and light latency-based evaluation.

### Pipeline Stages
Brand/Model Selection → Metadata Filtering → FAISS Vector Retrieval → Cross-Encoder
Re-ranking → Prompt Creation → LLM Generation → Grounded Answer with Attribution → Logging.
    
### Features
- Metadata-based filtering confines retrieval to the specified vehicle prior to any similarity
  search, decreasing false positives.
- Smart chunking by document sections (engine, safety, dimensions, comfort, and so on)
  is more semantically meaningful than plain size-based splitting.
- Re-ranking using cross-encoder is better at precision than just similarity provided by bi-encoder.
- Source information (document, section, page) allows for tracing and verification.
- Structured logging of per-query latencies at all pipeline stages for debugging.

### Drawbacks
- Classification of sections is based on keyword frequency. It is easy to understand but may
  mis-classify documents, which contain non-standard content and keywords.
- Currently, the chunking algorithm works on a per-page level. Very long or multi-topical
  pages are not split further which may decrease retrieval precision.
- Evaluation is done by latency only. Neither answer accuracy nor answer faithfulness was evaluated
  using automated framework in this notebook.
- The system is tested only on a few brochures. Retrieval quality at a larger scale is unknown.

### Potential Upgrades in Future
- Shift from page-wise to section-wise chunking in case of very long or detailed pages.
- Implement an evaluation of correctness/faithfulness based on frameworks like Ragas, once curated
  ground truth data is available for many models.
- Augment the list of keywords in section headings based on classification errors in other brochures.
- Cache embedding vectors and FAISS index for each brochure in order to save processing time.
- Enhance the Streamlit app with comparison functionality for multiple models.